# SFT->DPO training on ChartQA (method 7/7, final)

Full-trajectory DPO initialized from the ChartQA SFT LoRA (frozen SFT as the
DPO reference), same as CharXiv's SFT->DPO run -- only the dataset and SFT init
adapter change. The SFT adapter is gitignored (like all adapters in this repo),
so it's supplied via a private Kaggle Dataset (`chartqa-sft-adapter`) uploaded
after method 1/7 finished training, mirroring how the original project's
kaggle_train_sft_dpo kernel resolves its SFT adapter from a Kaggle Dataset mount
rather than git.


In [ ]:
import subprocess, sys

gpu_names = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().splitlines()
print('Detected GPUs (nvidia-smi):', gpu_names)
is_p100 = any('P100' in n for n in gpu_names)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

if is_p100:
    print('*** Tesla P100 detected. Installing the validated older stack...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow'], check=True)
else:
    print('Non-P100 GPU: upgrading transformers to a current release, leaving torch/peft/accelerate at image defaults.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.49.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft==0.14.0', 'qwen-vl-utils==0.0.14'], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, transformers: {transformers.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
import glob
import os
from pathlib import Path
import subprocess

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

pairs_path = repo_dir / 'experiments/020_chartqa_transfer/data/dpo_pairs.jsonl'
assert pairs_path.exists(), f'Missing {pairs_path} -- commit/push problem.'

# Locate the SFT ChartQA adapter by search rather than a hardcoded mount path --
# the same lesson learned during rollout generation earlier this session.
print('Contents of /kaggle/input:', os.listdir('/kaggle/input'))
matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
print('Found adapter_config.json at:', matches)
assert matches, 'SFT ChartQA adapter not found anywhere under /kaggle/input'
sft_adapter_dir = os.path.dirname(matches[0])
print('Using SFT init adapter:', sft_adapter_dir)
print('Adapter files:', sorted(os.listdir(sft_adapter_dir)))


In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_dpo.py',
    '--sft-dpo',
    '--init-adapter', sft_adapter_dir,
    '--dataset-path', 'experiments/020_chartqa_transfer/data/dpo_pairs.jsonl',
    '--images-dir', 'data/ChartQA/images',
    '--output-dir', '/kaggle/working/qwen_vl_sft_dpo_chartqa_adapter',
    '--epochs', '1',
    '--batch-size', '1',
    '--lr', '2e-6',
    '--beta', '0.1',
    '--max-logp-drop', '70',
    '--collapse-guard-warn-only',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, env=env, check=True)


In [ ]:
out_dir = Path('/kaggle/working/qwen_vl_sft_dpo_chartqa_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'SFT->DPO ChartQA adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- training did not save a LoRA adapter.'
